# RSNA Knee Abnormality Detection - Inference (Study-Level with MedDINOv3)

Processes all series for a study in a single forward pass.

In [1]:
# !pip install pydicom timm albumentations

import os, cv2, math, random, warnings
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm
from tqdm import tqdm
import multiprocessing as mp
from scipy.ndimage import zoom
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

os.environ['TORCH_HOME'] = '/tmp/torch_cache'

warnings.filterwarnings('ignore')
print('✓ All imports successful')

✓ All imports successful


In [2]:
"""
Simplified KneeAbnormalityModel using Dino backbone.

Architecture:
- Dino (frozen) extracts features from high-res images using 14x14 patches
- Features are mean-pooled to get per-slice embedding
- Slice transformer processes multi-slice sequence
- Metadata fusion and classification head
"""

import os
import pathlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from pathlib import Path
from typing import Optional

from rad_dino.utils import safetensors_to_state_dict


class DinoBackbone(nn.Module):
    """
    Wrapper around Dino model.
    Extracts features from images using frozen pretrained weights.
    Uses 14x14 patches natively.
    """
    def __init__(self, checkpoint_path: str, freeze_weights: bool = True):
        super().__init__()

        # Load base ViT-B model
        torch.hub.set_dir("/kaggle/input/models/mahyardana/kneefirst/pytorch/default/4/Archive")
        self.vit = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", pretrained=False)

        self.feat_dim = self.vit.embed_dim  # 768 for ViT-B

        # Freeze weights if requested
        if freeze_weights:
            for param in self.vit.parameters():
                param.requires_grad = False

    def _load_dino_weights(self, checkpoint_path: str):
        """Load Dino weights into ViT-B backbone."""
        if not os.path.exists(checkpoint_path):
            print(f"⚠ Warning: Dino checkpoint not found at {checkpoint_path}")
            print(f"  Using random initialization instead")
            return
        backbone_state_dict = safetensors_to_state_dict(checkpoint_path)

        # Remove 'backbone.' prefix if present
        # Load into ViT
        missing, unexpected = self.vit.load_state_dict(backbone_state_dict, strict=True)
        print(f"✓ Dino loaded (frozen)")
        print(f"  Missing keys: {len(missing)}")
        print(f"  Unexpected keys: {len(unexpected)}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through Dino backbone.

        Args:
            x: (B, 3, H, W) image tensor

        Returns:
            features: (B, N+1, 768) where N is number of patches
                     Includes CLS token as first position
        """
        # Use ViT's forward_features to get all patch tokens (including CLS)
        return self.vit.forward_features(x)  # (B, N+1, 768)


class MHSA(nn.Module):
    """Multi-head self-attention with optional key-padding mask."""
    def __init__(self, dim: int, heads: int = 8, dropout: float = 0.0):
        super().__init__()
        assert dim % heads == 0
        self.heads     = heads
        self.scale     = (dim // heads) ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(dropout)
        self.proj      = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        B, N, D = x.shape
        H = self.heads
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.reshape(B, N, H, D // H).transpose(1, 2)
        k = k.reshape(B, N, H, D // H).transpose(1, 2)
        v = v.reshape(B, N, H, D // H).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        attn = torch.nan_to_num(attn.softmax(dim=-1), nan=0.0)
        attn = self.attn_drop(attn)
        out  = torch.matmul(attn, v).transpose(1, 2).reshape(B, N, D)
        return self.proj_drop(self.proj(out))


class TransformerBlock(nn.Module):
    """Transformer block with pre-norm architecture."""
    def __init__(self, dim, heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MHSA(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.norm1(x), key_padding_mask)
        x = x + self.mlp(self.norm2(x))
        return x


class SliceTransformerEncoder(nn.Module):
    """Transformer encoder for processing slices."""
    def __init__(self, dim, heads, num_layers, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_ratio=4.0, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, src_key_padding_mask=None):
        for block in self.blocks:
            x = block(x, key_padding_mask=src_key_padding_mask)
        return self.norm(x)


class MetadataModule(nn.Module):
    """Per-slice metadata: fluid + fat + plane embeddings summed into one vector."""
    def __init__(self, num_planes: int, embed_dim: int):
        super().__init__()
        self.plane_embed = nn.Embedding(num_planes, embed_dim)
        self.fluid_embed = nn.Embedding(2, embed_dim)
        self.fat_embed   = nn.Embedding(2, embed_dim)
        for emb in (self.plane_embed, self.fluid_embed, self.fat_embed):
            nn.init.normal_(emb.weight, std=0.02)

    def forward(self, plane_id, fluid_sensitive, fat_suppression):
        # inputs can be (B, S)
        return (self.fluid_embed(fluid_sensitive) +
                self.fat_embed  (fat_suppression) +
                self.plane_embed(plane_id))


class KneeAbnormalityModelWithDino(nn.Module):
    """
    Simplified RSNA Knee model using frozen Dino.

    Processes ALL series of a study simultaneously.

    Pipeline:
    All Slices of Study (padded to multiple of 14)
      → Dino (frozen) → per-slice feature
      → fuse with per-slice metadata (plane/fluid/fat)
      → StudyTransformerEncoder (attend across ALL slices of ALL series)
      → mean-pool over slices
      → MLP → 12 logits
    """
    MAX_SLICES = 128  # Increased to handle all slices across multiple series

    def __init__(self, cfg, dino_checkpoint: Optional[str] = None):
        super().__init__()
        self.cfg = cfg

        self.backbone = DinoBackbone(dino_checkpoint, freeze_weights=False)
        backbone_feat_dim = self.backbone.feat_dim  # 768
        self.conv=nn.Sequential(
         nn.Conv1d(backbone_feat_dim,512, kernel_size=2),
            nn.MaxPool1d(2),
            nn.ReLU(),
         nn.Conv1d(512, 256, kernel_size=2),
            nn.MaxPool1d(2),
            nn.ReLU()
        )
        self.all_feat_encoder = SliceTransformerEncoder(
            dim=256+8,
            heads=8,
            num_layers=2,
            dropout=0.1
            )
        self.all_feat_encoder_layer2 = nn.TransformerEncoderLayer(
            d_model=256+8,
            nhead=8,
            activation='gelu',
            dropout=0.1,
            batch_first=True)
        self.all_feat_encoder2=nn.TransformerEncoder(
            encoder_layer=self.all_feat_encoder_layer2,
            num_layers=2,

        )
        self.meta_proj = nn.Linear(3, 8)


        num_labels = cfg.NUM_LABELS if hasattr(cfg, 'NUM_LABELS') else 12
        dropout = cfg.DROPOUT if hasattr(cfg, 'DROPOUT') else 0.3

        self.classifier = nn.Sequential(
            nn.LayerNorm(256+8), nn.Dropout(dropout),
            nn.Linear(256+8, 64), nn.ReLU(),
            nn.Dropout(dropout / 2), nn.Linear(64, num_labels))

    def train(self, mode: bool = True):
        """Override train to keep Dino frozen."""
        super().train(mode)
        # self.backbone.eval()
        # self.backbone.requires_grad_(False)
        return self

    def forward(self, images, slice_mask, plane_ids, fluid_sensitive, fat_suppression):
        """
        Forward pass through the model.

        Args:
            images: (B, Total_S, 3, H, W) - batch, slices across ALL series
                   Images should be padded to nearest multiple of 14
            slice_mask: (B, Total_S) bool - True = padding slice
            plane_ids: (B, Total_S) - anatomical plane ID
            fluid_sensitive: (B, Total_S) - fluid-sensitive sequence
            fat_suppression: (B, Total_S) - fat-suppressed sequence

        Returns:
            logits: (B, 12) - predictions for 12 labels
        """
        B, Serie,S, C, H, W= images.shape
        flat_images = images.view(B * Serie * S, C, H, W)

        # 1. Micro-chunk the backbone pass to prevent 4GB+ VRAM allocation spikes
        CHUNK_SIZE = 32  # Process max 32 images at a time through Dino
        dino_tokens = []

        with torch.no_grad():
            for i in range(0, flat_images.size(0), CHUNK_SIZE):
                chunk = flat_images[i: i + CHUNK_SIZE]
                chunk_out = self.backbone(chunk)["x_norm_clstoken"]
                dino_tokens.append(chunk_out)
        dino_out = torch.cat(dino_tokens, dim=0)
        # with torch.no_grad():
        #     dino_out = self.backbone(images.view(B*Serie*S,C, H, W))["x_norm_clstoken"]
        # cls_token = self.feat_pos.expand(B*Serie, -1, -1).view(B,Serie,1, -1)
        dino_out = dino_out.view(B,Serie,S, 768)
        # feat_with_cls = torch.cat((cls_token, dino_out), dim=2).view(B*Serie, -1, 768).transpose(1, 2)
        dino_out=dino_out.view(B*Serie, -1, 768).transpose(1, 2)
        convout=self.conv(dino_out)
        # featout=self.feat_encoder(feat_with_cls)[:, 0, :]
        convout2=convout.view(B,Serie, -1)
        meta_cat=torch.cat([plane_ids.unsqueeze(-1), fluid_sensitive.unsqueeze(-1),
                   fat_suppression.unsqueeze(-1)], dim=-1).view(B*Serie, -1)
        meta_cat=self.meta_proj(meta_cat).view(B,Serie, -1)

        meta_feat=torch.cat([convout2,meta_cat],-1)

        meta_feat=self.all_feat_encoder2(meta_feat).mean(1)
        return self.classifier(meta_feat)


def compute_valid_size(H: int, W: int, patch_size: int = 14) -> tuple:
    """
    Compute padded size for Dino (uses 14x14 patches).

    Args:
        H, W: Original image height and width
        patch_size: DINO patch size (14 for Dino)

    Returns:
        (padded_H, padded_W): Smallest size >= original satisfying size % patch_size == 0
    """
    pad_h = ((H + patch_size - 1) // patch_size) * patch_size
    pad_w = ((W + patch_size - 1) // patch_size) * patch_size
    return pad_h, pad_w


def pad_image(img: torch.Tensor, patch_size: int = 14) -> torch.Tensor:
    """
    Pad image to nearest multiple of patch_size.

    Args:
        img: (C, H, W), (B, C, H, W), or (B, S, C, H, W)
        patch_size: DINO patch size (14 for Dino)

    Returns:
        Padded image with same number of dimensions
    """
    if img.ndim == 3:
        C, H, W = img.shape
        pad_h, pad_w = compute_valid_size(H, W, patch_size)
        pad_top = (pad_h - H) // 2
        pad_bottom = pad_h - H - pad_top
        pad_left = (pad_w - W) // 2
        pad_right = pad_w - W - pad_left
        return F.pad(img.unsqueeze(0), (pad_left, pad_right, pad_top, pad_bottom), value=0.0).squeeze(0)
    elif img.ndim == 4:
        B, C, H, W = img.shape
        pad_h, pad_w = compute_valid_size(H, W, patch_size)
        pad_top = (pad_h - H) // 2
        pad_bottom = pad_h - H - pad_top
        pad_left = (pad_w - W) // 2
        pad_right = pad_w - W - pad_left
        return F.pad(img, (pad_left, pad_right, pad_top, pad_bottom), value=0.0)
    elif img.ndim == 5:
        B, S, C, H, W = img.shape
        pad_h, pad_w = compute_valid_size(H, W, patch_size)
        pad_top = (pad_h - H) // 2
        pad_bottom = pad_h - H - pad_top
        pad_left = (pad_w - W) // 2
        pad_right = pad_w - W - pad_left
        # F.pad pads the last dimensions. (left, right, top, bottom) pads W and H
        return F.pad(img, (pad_left, pad_right, pad_top, pad_bottom), value=0.0)
    else:
        raise ValueError(f"Unsupported image dimensions: {img.ndim}")

In [3]:
class InfConfig:
    KAGGLE_DIR      = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection')
    PYTHON_DIR      = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection')
    TEST_SERIES_CSV = KAGGLE_DIR / 'test_series.csv'
    TEST_IMG_DIR    = KAGGLE_DIR / 'test_series'
    CHECKPOINT      = str(PYTHON_DIR / 'best_model_raddino.pth') # Or update to best_model_meddinov3.pth
    SUBMISSION_PATH = str(KAGGLE_DIR / 'submission.csv')
    MAX_SLICES_PER_SERIES = 8
    MAX_SERIES_PER_STUDY = 16
    LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
              'Medial OA','Lateral OA','PF OA','Effusion',
              'Synovitis',"Baker's",'Contusion','Fracture']
    NUM_LABELS   = 12
    PLANE_TO_IDX = {'Sagittal': 0, 'Axial': 1, 'Coronal': 2}
    NUM_PLANES   = 3

    # Model dimensions
    SLICE_DIM    = 512
    META_DIM     = 256
    SLICE_ENC_HEADS  = 8
    SLICE_ENC_LAYERS = 2
    DROPOUT      = 0.3

    BATCH_SIZE   = 4
    NUM_WORKERS  = 4
    USE_TTA      = False
    DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

icfg = InfConfig()
print(f'Device: {icfg.DEVICE}')

Device: cuda


In [4]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)

In [5]:
# ── Test Dataset ──────────────────────────────────────────────────────────────
class KneeStudyDataset(Dataset):
    '''
    Groups all series for a study together.
    Returns a list of slices and their corresponding metadata.
    '''
    def __init__(self, df_series: pd.DataFrame, img_dir: Path, cfg, flip_tta: bool = False):
        self.cfg = cfg
        self.img_dir = img_dir
        self.flip_tta = flip_tta

        self.studies = []
        for study_id, group in df_series.groupby('StudyInstanceUID'):
            series_list = []
            for _, row in group.iterrows():
                series_list.append({
                    'series_id': str(row['SeriesInstanceUID']),
                    'plane_id': int(cfg.PLANE_TO_IDX.get(str(row['Anatomical_Plane']), 0)),
                    'fluid_sensitive': int(row.get('Fluid_Sensitive', 0)),
                    'fat_suppression': int(row.get('Fat_Suppression', 0))
                })
            self.studies.append({'study_id': str(study_id), 'series': series_list})

    def __len__(self):
        return len(self.studies)

    @staticmethod
    def resize_shorter_side_scipy(img, target_short=518):
        h, w = img.shape[:2]

        # Calculate scale factor
        scaleh = target_short / h
        scalew = target_short / w

        # Check if image has channels (e.g., 3D array for RGB or 2D for Grayscale DICOM)
        if len(img.shape) == 3:
            zoom_factors = (scaleh, scalew, 1)  # Don't scale channels
        else:
            zoom_factors = (scaleh, scalew)

        # order=3 specifies a Cubic B-Spline interpolation
        resized_img = zoom(img, zoom_factors, order=3)
        return resized_img
    @staticmethod
    def safe_load_dicom_array(dcm_path: str):
        """
        Handles uncompressed (Explicit/Implicit VR) and compressed
        (JPEG Lossless, JPEG 2000) DICOM transfer syntaxes automatically.
        """
        try:
            ds = pydicom.dcmread(dcm_path)
            # Forces pylibjpeg/openjpeg decompression backend
            arr = ds.pixel_array.astype(np.float32)

            # Check for MONOCHROME1 pixel inversion
            if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
                arr = arr.max() - arr

            return arr
        except Exception as e:
            # Fallback or corrupted DICOM handling
            return None
    def __getitem__(self, idx):
        study = self.studies[idx]
        study_id = study['study_id']

        all_slices = []
        all_plane_ids = []
        all_fluids = []
        all_fats = []
        for sinfo in study['series']:
            path = self.img_dir / study_id / sinfo['series_id']
            if not path.exists():
                continue
            dcm_files = []
            alldcms=[pydicom.dcmread(str(dcm),stop_before_pixels=True) for dcm in sorted(path.glob('*.dcm'))]
            # Filter out files missing critical spatial metadata
            valid_slices = [
                ds for ds in alldcms
                if "ImagePositionPatient" in ds and "ImageOrientationPatient" in ds
            ]

            if not valid_slices:
                raise ValueError("No valid spatial metadata found in the provided DICOM slices.")

            # Extract orientation vectors from the first slice
            orient = valid_slices[0].ImageOrientationPatient
            row_cosines = np.array(orient[:3])
            col_cosines = np.array(orient[3:])

            # Compute the normal vector (orthogonal to the slice plane)
            slice_normal = np.cross(row_cosines, col_cosines)

            # Sort based on the projection of ImagePosition onto the slice normal
            sorted_slices = sorted(
                valid_slices,
                key=lambda ds: np.dot(np.array(ds.ImagePositionPatient), slice_normal)
            )
            alldcms=sorted_slices
            if self.cfg.MAX_SLICES_PER_SERIES>0:
                skipnum = math.ceil(len(alldcms) / self.cfg.MAX_SLICES_PER_SERIES)
                for i in range(len(alldcms)):
                    if (i % skipnum) == 0:
                        if i==0:
                            dcm_files.append(alldcms[i])
                            dcm_files.append(alldcms[i+1])
                            dcm_files.append(alldcms[i+2])
                        elif i==len(alldcms)-1:
                            dcm_files.append(alldcms[i-2])
                            dcm_files.append(alldcms[i-1])
                            dcm_files.append(alldcms[i])
                        else:
                            dcm_files.append(alldcms[i-1])
                            dcm_files.append(alldcms[i])
                            dcm_files.append(alldcms[i+1])
            else:
                dcm_files = alldcms
            slices=[]
            tempimgs=[]
            for dcm_file in dcm_files:
                # dcm = pydicom.dcmread(str(dcm_file))
                dcm=dcm_file
                try:
                    img = self.safe_load_dicom_array(dcm.filename)
                    img = (img - img.min()) / (img.max() - img.min() + 1e-6)

                    # Very simple augmentations for training
                    # if self.is_train and random.random() > 0.5:
                    #     img = np.fliplr(img).copy()
                    # cv2.resize
                    if img.shape[0]!=img.shape[1]:
                        pass
                    # padded_img = cv2.copyMakeBorder(
                    #     img,
                    #     0,
                    #     bottom,
                    #     0,
                    #     right,
                    #     borderType=cv2.BORDER_CONSTANT,
                    #     value=[0, 0, 0]
                    # )
                    # img = self.resize_shorter_side_scipy(img)
                    img = cv2.resize(img, (518, 518), interpolation=cv2.INTER_CUBIC) # Base size, will be padded natively
                    img = np.clip(img, 0.0, 1.0)
                    # img = np.stack([img, img, img], axis=0)  # (3, H, W)
                    tempimgs.append(torch.from_numpy(img))
                except:
                    pass
                if len(tempimgs)==3:
                    slices.append(torch.stack(tempimgs, dim=0))
                    tempimgs=[]
            if len(tempimgs)>0:
                toadd=3-len(tempimgs)
                newtempimgs=torch.cat([torch.stack(tempimgs,dim=0),torch.zeros(toadd, 518, 518)],dim=0)
                slices.append(newtempimgs)

            all_plane_ids.append(sinfo['plane_id'])
            all_fluids.append(sinfo['fluid_sensitive'])
            all_fats.append(sinfo['fat_suppression'])
            all_slices.append(torch.stack(slices))

        if len(all_slices) == 0:
            # Fallback if no images found
            all_slices.append(torch.zeros(3, 518, 518))
            all_plane_ids.append(0)
            all_fluids.append(0)
            all_fats.append(0)
        max_slice=max(slices.shape[0] for slices in all_slices)
        max_height=max(slices.shape[2] for slices in all_slices)
        max_width=max(slices.shape[3] for slices in all_slices)
        if max_height%14!=0:
            max_height+=14-(max_height%14)
        if max_width%14!=0:
            max_width+=14-(max_width%14)
        if max_width>1000 or max_height>1000:
            pass

        for idx in range(len(all_slices)):
            if all_slices[idx].shape[2]<max_height or all_slices[idx].shape[3]<max_width:
                pad_height=max_height - all_slices[idx].shape[2]
                pad_width=max_width - all_slices[idx].shape[3]
                all_slices[idx]=F.pad(all_slices[idx], (0, pad_width, 0, pad_height),value=0, mode="constant")
            if all_slices[idx].shape[0] < max_slice:
                sizediff = max_slice - all_slices[idx].shape[0]
                all_slices[idx] = torch.cat([all_slices[idx], torch.zeros((sizediff, 3, max_height, max_width))],
                                            dim=0)

        return {
            'study_id': study_id,
            'slices': torch.stack(all_slices), # (S, 3, H, W)
            'plane_ids': torch.tensor(all_plane_ids, dtype=torch.float32),
            'fluid_sensitive': torch.tensor(all_fluids, dtype=torch.float32),
            'fat_suppression': torch.tensor(all_fats, dtype=torch.float32)
        }


In [6]:
def collate_fn(batch: List[Dict]) -> Dict:
    max_s = icfg.MAX_SERIES_PER_STUDY
    max_slice = icfg.MAX_SLICES_PER_SERIES#max(b['slices'].shape[1] for b in batch)#cfg.MAX_SLICES_PER_SERIES
    max_height = max(b['slices'].shape[-2] for b in batch)
    max_width = max(b['slices'].shape[-1] for b in batch)
    padded_slices = []
    padded_planes = []
    padded_fluids = []
    padded_fats = []
    slice_masks = []
    study_ids = []
    placeholder_planes=10.0
    placeholder_fluids=10.0
    placeholder_fats=10.0
    for b in batch:
        s = b['slices'].shape[0]
        slice_len=b['slices'].shape[1]
        study_ids.append(b['study_id'])
        padded_slice = b['slices']
        if padded_slice.shape[-2] < max_height or padded_slice.shape[-1] < max_width:
            pad_height = max_height - padded_slice.shape[-2]
            pad_width = max_width - padded_slice.shape[-1]
            padded_slice = F.pad(padded_slice, (0, pad_width, 0, pad_height), value=0, mode="constant")
        if slice_len<max_slice:
            sizediff = max_slice - slice_len
            padded_slice = torch.cat([padded_slice, torch.zeros((s,sizediff, 3, max_height, max_width))],dim=1)
        if s < max_s:
            pad_img = torch.zeros(max_s - s, *padded_slice.shape[1:])
            padded_slices.append(torch.cat([padded_slice, pad_img], dim=0))
            pad_idx = torch.ones(max_s - s)
            padded_planes.append(torch.cat([b['plane_ids'], pad_idx*placeholder_planes], dim=0))
            padded_fluids.append(torch.cat([b['fluid_sensitive'], pad_idx*placeholder_fluids], dim=0))
            padded_fats.append(torch.cat([b['fat_suppression'], pad_idx*placeholder_fats], dim=0))

            mask = torch.cat([torch.zeros(s, dtype=torch.bool), torch.ones(max_s - s, dtype=torch.bool)])
            slice_masks.append(mask)
        else:
            padded_slices.append(padded_slice)
            padded_planes.append(b['plane_ids'])
            padded_fluids.append(b['fluid_sensitive'])
            padded_fats.append(b['fat_suppression'])
            slice_masks.append(torch.zeros(s, dtype=torch.bool))
        # max_slice = max(slices.shape[0] for slices in all_slices)
        # max_height = max(slices.shape[2] for slices in all_slices)
        # max_width = max(slices.shape[3] for slices in all_slices)
        # if max_height % 14 != 0:
        #     max_height += 14 - (max_height % 14)
        # if max_width % 14 != 0:
        #     max_width += 14 - (max_width % 14)
        #
        # for idx in range(len(all_slices)):
        #     if all_slices[idx].shape[2] < max_height or all_slices[idx].shape[3] < max_width:
        #         pad_height = max_height - all_slices[idx].shape[2]
        #         pad_width = max_width - all_slices[idx].shape[3]
        #         all_slices[idx] = F.pad(all_slices[idx], (0, pad_width, 0, pad_height), value=0, mode="constant")
        #     if all_slices[idx].shape[0] < max_slice:
        #         sizediff = max_slice - all_slices[idx].shape[0]
        #         all_slices[idx] = torch.cat([all_slices[idx], torch.zeros((sizediff, 3, max_height, max_width))],
        #                                     dim=0)
    # images = [[torch.stack(padded_slice) for padded_slice in padded_slice_outer] for padded_slice_outer in padded_slices] # (B, max_s, 3, 224, 224)
    # Pad natively for RadDino to 14x multiples
    images = padded_slices # (B, max_s, 3, H, W)

    return {
        'study_ids': study_ids,
        'images': torch.stack(images),
        'slice_mask': torch.stack(slice_masks),
        'plane_ids': torch.stack(padded_planes),
        'fluid_sensitive': torch.stack(padded_fluids),
        'fat_suppression': torch.stack(padded_fats)
    }

In [7]:
# ── Load Model ────────────────────────────────────────────────────────────────
print('Loading model...')
model = KneeAbnormalityModelWithDino(icfg).to(icfg.DEVICE)

if os.path.exists(icfg.CHECKPOINT):
    ckpt = torch.load(icfg.CHECKPOINT, map_location='cpu')
    if 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)
    print(f"✓ Checkpoint loaded from {icfg.CHECKPOINT}")
else:
    print(f"⚠ Checkpoint not found: {icfg.CHECKPOINT}")

model.eval()
print(f"✓ Model on device: {icfg.DEVICE}")

Loading model...


PermissionError: [Errno 13] Permission denied: '/kaggle'

In [ ]:
@torch.no_grad()
def run_inference(model, loader, device):
    model.eval()
    study_ids, all_probs = [], []
    
    for batch in tqdm(loader, desc='Inference'):
        images = batch['images'].to(device)
        slice_mask = batch['slice_mask'].to(device)
        plane_ids = batch['plane_ids'].to(device)
        fluid_sensitive = batch['fluid_sensitive'].to(device)
        fat_suppression = batch['fat_suppression'].to(device)
        # with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_probs.append(probs)
        study_ids.extend(batch['study_ids'])
    
    return study_ids, np.vstack(all_probs)

print('✓ Inference function ready')

In [ ]:
# ── Run Inference ─────────────────────────────────────────────────────────────
df_test_series = pd.read_csv(icfg.TEST_SERIES_CSV)
print(f'Test series: {len(df_test_series)}')
print(f'Test studies: {df_test_series["StudyInstanceUID"].nunique()}')
print('Planes:', df_test_series['Anatomical_Plane'].value_counts().to_dict())

ds_orig = KneeStudyDataset(df_test_series, icfg.TEST_IMG_DIR, icfg, flip_tta=False)
ld_orig = DataLoader(ds_orig, batch_size=icfg.BATCH_SIZE, shuffle=False,
                     num_workers=icfg.NUM_WORKERS, pin_memory=True,
                     collate_fn=collate_fn)

sids, probs_orig = run_inference(model, ld_orig, icfg.DEVICE)

if icfg.USE_TTA:
    ds_flip = KneeStudyDataset(df_test_series, icfg.TEST_IMG_DIR, icfg, flip_tta=True)
    ld_flip = DataLoader(ds_flip, batch_size=icfg.BATCH_SIZE, shuffle=False,
                         num_workers=icfg.NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_fn)
    _, probs_flip = run_inference(model, ld_flip, icfg.DEVICE)
    probs_all = (probs_orig + probs_flip) / 2.0
    print("TTA applied (original + h-flip averaged)")
else:
    probs_all = probs_orig
    
print(f'\nGenerated predictions for {len(sids)} studies')
print(f'Probabilities shape: {probs_all.shape}')

In [ ]:
# ── Create Submission ─────────────────────────────────────────────────────────
rows = [{'StudyInstanceUID': sid, **dict(zip(icfg.LABELS, p))}
        for sid, p in zip(sids, probs_all)]
df_sub = pd.DataFrame(rows)


df_sub.to_csv(icfg.SUBMISSION_PATH, index=False)
print(f'✓ Submission saved to {icfg.SUBMISSION_PATH}')
print(df_sub.head())

In [ ]:
exit()